# 03 — Matched Transfer Experiments: Baseline vs External Features

This notebook is run after `01_Phase2_Core_Final.ipynb` and `02_External_Features_Final.ipynb`.

Corrections:
- The singularity of `UrbanId-year` is validated before merging (`validate='one_to_one'`).
- Target country test cities are kept **constant between ratios** in each analysis pool. - Few-shot tagged cities are only taken from the adaptation pool; test cities never enter training. - Baseline and extended models are compared on the same pool, test cities, and seed/ratio.
- StandardScaler and imputation for Ridge are only within the pipeline fitted to the training data.
- The smaller subset containing NDVI is reported as a separate and matched sensitivity experiment. - Result metrics are not declared "successful" based solely on file existence; manual data QA validation is required.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib, warnings, shutil
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DRIVE_ROOT=Path('/content/drive/MyDrive')
_REPO_CANDIDATES=[
    DRIVE_ROOT / 'Urban_Heat_Transfer_COMPLETE_BUNDLE' / 'urban-heat-transfer',
    DRIVE_ROOT / 'urban-heat-transfer',
    DRIVE_ROOT / 'BIRLESIK_PROJE_PAKETI' / 'urban-heat-transfer',
]
PROJECT_DIR=next((p for p in _REPO_CANDIDATES if (p/'notebooks').is_dir() and (p/'data').is_dir()),None)
LEGACY_LAYOUT=PROJECT_DIR is None
if PROJECT_DIR is None:
    PROJECT_DIR=DRIVE_ROOT/'tao_documents' if (DRIVE_ROOT/'tao_documents').is_dir() else DRIVE_ROOT
if LEGACY_LAYOUT:
    PHASE2_ROOT=PROJECT_DIR/'phase2_outputs'
    EXTERNAL_DIR=PROJECT_DIR/'external_features'
    BACKUP_ROOT=PROJECT_DIR/'previous_output_backups'
else:
    PHASE2_ROOT=PROJECT_DIR/'outputs'/'phase2_outputs'
    EXTERNAL_DIR=PROJECT_DIR/'data'/'external_features'
    BACKUP_ROOT=PROJECT_DIR/'outputs'/'previous_output_backups'
DOCS_DIR=PROJECT_DIR/'docs'
for p in [PHASE2_ROOT,DOCS_DIR]: p.mkdir(parents=True,exist_ok=True)
PRE_RUN_BACKUP_DIR=BACKUP_ROOT/f'transfer_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
transfer_output_names=['task6_feature_matrix_enhanced.csv','task6_common_pool_coverage.csv','task6_enhanced_feature_coverage_by_country.csv',
                       'task7_transfer_results_all_variants.csv','task7_transfer_results_primary.csv','task7_transfer_results_enhanced.csv',
                       'task7_split_manifest_enhanced.csv','task7_few_shot_selections_enhanced.csv','baseline_vs_enhanced_comparison.csv',
                       'task7_partition_report.md','DECISION_LOG_ENHANCED.md']
old_transfer_outputs=[PHASE2_ROOT/name for name in transfer_output_names if (PHASE2_ROOT/name).exists()]
old_transfer_outputs += [DOCS_DIR/'DECISION_LOG_ENHANCED.md'] if (DOCS_DIR/'DECISION_LOG_ENHANCED.md').exists() else []
if old_transfer_outputs:
    PRE_RUN_BACKUP_DIR.mkdir(parents=True,exist_ok=True)
    for old_path in old_transfer_outputs:
        dest=PRE_RUN_BACKUP_DIR/old_path.relative_to(PROJECT_DIR)
        dest.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(old_path,dest)
    print(f'Archived {len(old_transfer_outputs):,} previous transfer outputs to {PRE_RUN_BACKUP_DIR}')
RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')
YEAR_MIN,YEAR_MAX=2001,2020
TARGET_NAME='Intensity_MEA'
FEW_SHOT_RATIOS=[0.0,0.01,0.05,0.10,0.20,0.50]
ADAPTATION_SEEDS=list(range(10))
TEST_FRACTION=0.20
TEST_LOCK_SEED=20261017

base_path=PHASE2_ROOT/'task6_feature_matrix.csv'
ext_path=EXTERNAL_DIR/'external_features.csv'
if not base_path.exists(): raise FileNotFoundError(f'Run 01_Phase2_Core_Final.ipynb first: {base_path}')
if not ext_path.exists(): raise FileNotFoundError(f'Run 02_External_Features_Final.ipynb first: {ext_path}')
base=pd.read_csv(base_path)
extf=pd.read_csv(ext_path)
if base.duplicated(['UrbanId','year']).any(): raise ValueError('Baseline feature matrix has duplicate UrbanId-year keys.')
if extf.duplicated(['UrbanId','year']).any(): raise ValueError('External features table has duplicate UrbanId-year keys.')
required_base={'UrbanId','year','Intensity_MEA','Country','Longitude','Latitude','Area','area_log'}
if not required_base.issubset(base.columns): raise ValueError(f'Baseline table missing columns: {required_base-set(base.columns)}')

# Normalize prior/legacy naming once; canonical output names are used below.
rename_map={}
if 'elevation_srtm' in extf.columns and 'elevation_m' not in extf.columns: rename_map['elevation_srtm']='elevation_m'
if 'era5_precip_jja' in extf.columns and 'era5_precip_jja_mm' not in extf.columns: rename_map['era5_precip_jja']='era5_precip_jja_mm'
extf=extf.rename(columns=rename_map)

joined=base.merge(extf.drop(columns=['Country','Longitude','Latitude','Area'],errors='ignore'),
                  on=['UrbanId','year'],how='left',validate='one_to_one')
if len(joined)!=len(base): raise AssertionError('External merge changed the baseline target row count.')
if joined.duplicated(['UrbanId','year']).any(): raise AssertionError('Merged feature matrix has duplicate city-year keys.')

BASELINE_FEATURES=['year','Longitude','Latitude','area_log']
DEM_ERA5_FEATURES=['elevation_m','era5_t2m_jja','era5_precip_jja_mm','era5_wind_jja','era5_dewpoint_jja','era5_rh_jja']
NDVI_FEATURES=['ndvi_jja_mean']
forbidden={'UrbanId','Country','country','Intensity_MEA','Intensity_EA','Intensity_IEA','Intensity_DEA',
           'Intensity_MEA_jun','Intensity_MEA_jul','Intensity_MEA_aug','Jun_MEA','Jul_MEA','Aug_MEA'}
if set(BASELINE_FEATURES+DEM_ERA5_FEATURES+NDVI_FEATURES) & forbidden:
    raise AssertionError('A forbidden ID/country/target-derived field is present in the predictor list.')
missing_primary=set(DEM_ERA5_FEATURES)-set(joined.columns)
missing_ndvi=set(NDVI_FEATURES)-set(joined.columns)
if missing_primary: raise ValueError(f'Expected primary external features missing: {missing_primary}')
if missing_ndvi: warnings.warn(f'NDVI feature missing; NDVI ablation will be blocked: {missing_ndvi}')

joined.to_csv(PHASE2_ROOT/'task6_feature_matrix_enhanced.csv',index=False)

Mounted at /content/drive
Archived 11 previous transfer outputs to /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/previous_output_backups/transfer_20261019_204841


## Step 1 — Feature Coverage and Matched Samples

To allow for direct comparison of baseline and extended results, the baseline model is evaluated in the same city-year rows with complete external features. In the NDVI susceptibility experiment, the baseline and NDVI models are compared in the same subset where the NDVI is valid. This prevents confusion between “model difference” and “sample difference”.

In [ ]:
primary_required=BASELINE_FEATURES+DEM_ERA5_FEATURES+[TARGET_NAME,'Country']
primary_pool=joined.dropna(subset=primary_required).copy()
ndvi_pool=joined.dropna(subset=primary_required+NDVI_FEATURES).copy()
if primary_pool.empty: raise ValueError('Primary common pool is empty; inspect external feature coverage.')
if ndvi_pool.empty: warnings.warn('NDVI complete-case pool is empty; NDVI sensitivity models will not run.')

pools={'PRIMARY_COMMON_POOL':primary_pool}
if not ndvi_pool.empty: pools['NDVI_COMPLETE_SUBSET']=ndvi_pool
coverage_rows=[]
for pool_name,df in pools.items():
    for country,sub in df.groupby('Country'):
        coverage_rows.append({'pool':pool_name,'Country':country,'n_rows':len(sub),'n_cities':sub.UrbanId.nunique(),
                              'min_year':int(sub.year.min()),'max_year':int(sub.year.max())})
coverage=pd.DataFrame(coverage_rows)
coverage.to_csv(PHASE2_ROOT/'task6_common_pool_coverage.csv',index=False)

missing_rows=[]
for feature in BASELINE_FEATURES+DEM_ERA5_FEATURES+NDVI_FEATURES:
    if feature not in joined.columns: continue
    for country,sub in joined.groupby('Country'):
        missing_rows.append({'feature':feature,'Country':country,'n_rows':len(sub),'n_missing':int(sub[feature].isna().sum()),
                             'missing_pct':float(100*sub[feature].isna().mean()),'n_cities_with_any_value':int(sub.loc[sub[feature].notna(),'UrbanId'].nunique())})
pd.DataFrame(missing_rows).to_csv(PHASE2_ROOT/'task6_enhanced_feature_coverage_by_country.csv',index=False)

print('Baseline target rows:',len(base))
print('Primary common pool rows:',len(primary_pool),'cities:',primary_pool.UrbanId.nunique())
print('NDVI complete subset rows:',len(ndvi_pool),'cities:',ndvi_pool.UrbanId.nunique())
print(coverage.to_string(index=False))

Baseline target rows: 32420
Primary common pool rows: 32420 cities: 1621
NDVI complete subset rows: 18237 cities: 1490
                pool Country  n_rows  n_cities  min_year  max_year
 PRIMARY_COMMON_POOL   China   28820      1441      2001      2020
 PRIMARY_COMMON_POOL Türkiye    3600       180      2001      2020
NDVI_COMPLETE_SUBSET   China   14892      1313      2001      2020
NDVI_COMPLETE_SUBSET Türkiye    3345       177      2001      2020


## Step 2 — Fixed test cities and few-shot experiments

Test cities are selected once for each pool/direction. Few-shot ratios are applied to the target city pool that is not included in the test. Ten different adaptation seeds are used (a single run is sufficient for 0%). The ratio is the percentage of cities in the adaptation pool; it should not be interpreted as the percentage of all cities in the target country.

In [ ]:
def make_split(df, source_country, target_country, pool_name):
    source_ids=sorted(df.loc[df.Country.eq(source_country),'UrbanId'].astype(int).unique().tolist())
    target_ids=sorted(df.loc[df.Country.eq(target_country),'UrbanId'].astype(int).unique().tolist())
    if len(source_ids)<2 or len(target_ids)<2:
        raise ValueError(f'Insufficient city count in {pool_name} {source_country}->{target_country}: {len(source_ids)}->{len(target_ids)}')
    # Seed is fixed per target country and pool, independent of few-shot ratio or model variant.
    pool_offset=0 if pool_name=='PRIMARY_COMMON_POOL' else 50_000
    rng=np.random.default_rng(TEST_LOCK_SEED+pool_offset+(1 if target_country=='Türkiye' else 2))
    perm=rng.permutation(target_ids)
    n_test=max(1,int(round(TEST_FRACTION*len(target_ids))))
    test_ids=sorted(map(int,perm[:n_test]))
    adaptation_ids=sorted(set(target_ids)-set(test_ids))
    return source_ids,adaptation_ids,test_ids


def make_estimator(model_name,seed):
    if model_name=='source_mean':
        return DummyRegressor(strategy='mean')
    if model_name=='ridge':
        return Pipeline([('imputer',SimpleImputer(strategy='median')),
                         ('scaler',StandardScaler()),('model',Ridge(alpha=1.0))])
    if model_name=='rf':
        return Pipeline([('imputer',SimpleImputer(strategy='median')),
                         ('model',RandomForestRegressor(n_estimators=100,max_depth=10,random_state=seed,n_jobs=-1))])
    raise ValueError(model_name)


def score_frame(test_df,y_true,y_pred):
    residual=np.asarray(y_true)-np.asarray(y_pred)
    by_city=pd.DataFrame({'UrbanId':test_df.UrbanId.to_numpy(),'ae':np.abs(residual),'se':residual**2}).groupby('UrbanId').agg(city_mae=('ae','mean'),city_mse=('se','mean'))
    return {'r2':float(r2_score(y_true,y_pred)) if len(y_true)>1 else np.nan,
            'rmse':float(np.sqrt(mean_squared_error(y_true,y_pred))),
            'mae':float(mean_absolute_error(y_true,y_pred)),
            'macro_city_mae':float(by_city.city_mae.mean()),
            'macro_city_rmse':float(np.sqrt(by_city.city_mse.mean()))}

variants={
    'PRIMARY_COMMON_POOL': {'geo_baseline':BASELINE_FEATURES,'dem_era5':BASELINE_FEATURES+DEM_ERA5_FEATURES},
}
if not ndvi_pool.empty:
    variants['NDVI_COMPLETE_SUBSET']={'geo_baseline':BASELINE_FEATURES,
                                      'dem_era5':BASELINE_FEATURES+DEM_ERA5_FEATURES,
                                      'dem_era5_ndvi':BASELINE_FEATURES+DEM_ERA5_FEATURES+NDVI_FEATURES}

results=[]; split_rows=[]; selection_rows=[]
for pool_name, variant_dict in variants.items():
    df=pools[pool_name]
    for direction,source_country,target_country in [
        ('China_to_Turkiye','China','Türkiye'),('Turkiye_to_China','Türkiye','China')]:
        source_ids,adapt_ids,test_ids=make_split(df,source_country,target_country,pool_name)
        split_rows += [{'pool':pool_name,'direction':direction,'UrbanId':u,'role':'SOURCE_TRAIN','test_lock_seed':TEST_LOCK_SEED} for u in source_ids]
        split_rows += [{'pool':pool_name,'direction':direction,'UrbanId':u,'role':'TARGET_ADAPTATION_POOL','test_lock_seed':TEST_LOCK_SEED} for u in adapt_ids]
        split_rows += [{'pool':pool_name,'direction':direction,'UrbanId':u,'role':'TARGET_TEST_LOCKED','test_lock_seed':TEST_LOCK_SEED} for u in test_ids]
        assert not (set(source_ids)&set(adapt_ids) or set(source_ids)&set(test_ids) or set(adapt_ids)&set(test_ids))
        target_tests = set(test_ids)
        # Target test IDs are constructed once per pool/direction and reused for all variants/ratios/seeds.
        test_df=df[df.UrbanId.isin(target_tests)].copy()
        if set(test_df.UrbanId.unique()) != target_tests: raise AssertionError('Some locked test cities have no records in the pool.')
        for ratio in FEW_SHOT_RATIOS:
            seeds=[0] if ratio==0 else ADAPTATION_SEEDS
            for seed in seeds:
                if ratio==0: shots=[]
                else:
                    rng=np.random.default_rng(33000+seed+(0 if direction=='China_to_Turkiye' else 100)+(0 if pool_name=='PRIMARY_COMMON_POOL' else 1000))
                    n_shots=min(len(adapt_ids),max(1,int(round(len(adapt_ids)*ratio))))
                    shots=sorted(map(int,rng.permutation(np.asarray(adapt_ids, dtype=int))[:n_shots]))
                selection_rows.extend([{'pool':pool_name,'direction':direction,'ratio_of_adaptation_pool':ratio,'seed':seed,'UrbanId':u} for u in shots])
                train_ids=sorted(set(source_ids)|set(shots))
                assert not (set(train_ids)&target_tests), 'Test city leakage: locked test ID appears in training.'
                train_df=df[df.UrbanId.isin(train_ids)].copy()
                if train_df.empty or test_df.empty: raise ValueError(f'Empty train/test for {pool_name}, {direction}, ratio={ratio}')
                for feature_set,features in variant_dict.items():
                    if set(features)&forbidden: raise AssertionError(f'Forbidden feature in model: {set(features)&forbidden}')
                    for model_name in ['source_mean','ridge','rf']:
                        estimator=make_estimator(model_name,seed)
                        estimator.fit(train_df[features],train_df[TARGET_NAME])
                        pred=estimator.predict(test_df[features])
                        overlap=len(set(train_df.UrbanId.astype(int).unique()) & set(test_df.UrbanId.astype(int).unique()))
                        if overlap != 0:
                            raise AssertionError(f'Test city leakage in {pool_name}/{direction}: {overlap} overlapping cities')
                        results.append({'pool':pool_name,'direction':direction,'source_country':source_country,
                                        'target_country':target_country,'feature_set':feature_set,'model':model_name,
                                        'few_shot_ratio':ratio,'seed':seed,'n_source_cities':len(source_ids),
                                        'n_adaptation_pool_cities':len(adapt_ids),'n_shot_cities':len(shots),
                                        'n_test_cities':len(test_ids),'n_train_rows':len(train_df),'n_test_rows':len(test_df),
                                        'test_city_overlap':overlap,**score_frame(test_df,test_df[TARGET_NAME].to_numpy(),pred)})

results_all=pd.DataFrame(results)
if results_all.empty: raise RuntimeError('No transfer results were generated.')
results_all.to_csv(PHASE2_ROOT/'task7_transfer_results_all_variants.csv',index=False)
pd.DataFrame(split_rows).to_csv(PHASE2_ROOT/'task7_split_manifest_enhanced.csv',index=False)
pd.DataFrame(selection_rows,columns=['pool','direction','ratio_of_adaptation_pool','seed','UrbanId']).to_csv(PHASE2_ROOT/'task7_few_shot_selections_enhanced.csv',index=False)

primary_out=results_all[(results_all.pool=='PRIMARY_COMMON_POOL') & (results_all.feature_set=='dem_era5')].copy()
primary_out.to_csv(PHASE2_ROOT/'task7_transfer_results_primary.csv',index=False)
if 'NDVI_COMPLETE_SUBSET' in set(results_all.pool):
    ndvi_out=results_all[(results_all.pool=='NDVI_COMPLETE_SUBSET') & (results_all.feature_set=='dem_era5_ndvi')].copy()
else:
    ndvi_out=pd.DataFrame()
ndvi_out.to_csv(PHASE2_ROOT/'task7_transfer_results_enhanced.csv',index=False)

# Paired comparisons are made only within same pool/direction/model/ratio/seed, so test cities and rows match.
comparison_rows=[]
for pool_name, base_variant, enhanced_variant, label in [
    ('PRIMARY_COMMON_POOL','geo_baseline','dem_era5','baseline_vs_dem_era5'),
    ('NDVI_COMPLETE_SUBSET','dem_era5','dem_era5_ndvi','dem_era5_vs_ndvi')]:
    sub=results_all[results_all.pool.eq(pool_name)]
    if sub.empty: continue
    keys=['pool','direction','model','few_shot_ratio','seed']
    left=sub[sub.feature_set.eq(base_variant)][keys+['r2','rmse','mae','macro_city_mae','macro_city_rmse']].rename(columns={c:f'{c}_baseline' for c in ['r2','rmse','mae','macro_city_mae','macro_city_rmse']})
    right=sub[sub.feature_set.eq(enhanced_variant)][keys+['r2','rmse','mae','macro_city_mae','macro_city_rmse']].rename(columns={c:f'{c}_enhanced' for c in ['r2','rmse','mae','macro_city_mae','macro_city_rmse']})
    paired=left.merge(right,on=keys,how='inner',validate='one_to_one')
    if len(paired):
        paired['comparison']=label
        paired['delta_r2']=paired['r2_enhanced']-paired['r2_baseline']
        paired['delta_rmse']=paired['rmse_enhanced']-paired['rmse_baseline']
        paired['delta_mae']=paired['mae_enhanced']-paired['mae_baseline']
        comparison_rows.append(paired)
comparison=pd.concat(comparison_rows,ignore_index=True) if comparison_rows else pd.DataFrame()
comparison.to_csv(PHASE2_ROOT/'baseline_vs_enhanced_comparison.csv',index=False)

print('Primary results:',len(primary_out))
print('NDVI enhanced results:',len(ndvi_out))
print(results_all.groupby(['pool','feature_set','direction','model','few_shot_ratio'])[['r2','rmse','mae','macro_city_mae']].mean().reset_index().to_string(index=False))

Primary results: 306
NDVI enhanced results: 306
                pool   feature_set        direction       model  few_shot_ratio         r2     rmse      mae  macro_city_mae
NDVI_COMPLETE_SUBSET      dem_era5 China_to_Turkiye          rf            0.00   0.185979 0.653215 0.529086        0.521388
NDVI_COMPLETE_SUBSET      dem_era5 China_to_Turkiye          rf            0.01   0.251135 0.626062 0.509215        0.502868
NDVI_COMPLETE_SUBSET      dem_era5 China_to_Turkiye          rf            0.05   0.188676 0.650430 0.516484        0.511883
NDVI_COMPLETE_SUBSET      dem_era5 China_to_Turkiye          rf            0.10   0.239719 0.629491 0.500990        0.497998
NDVI_COMPLETE_SUBSET      dem_era5 China_to_Turkiye          rf            0.20   0.172197 0.655533 0.515440        0.513153
NDVI_COMPLETE_SUBSET      dem_era5 China_to_Turkiye          rf            0.50   0.101699 0.682855 0.510295        0.508058
NDVI_COMPLETE_SUBSET      dem_era5 China_to_Turkiye       ridge            0.

## Step 3 — The True GO/NO-GO

This decision not only checks for the existence of necessary files; it also checks the structure of data keys, split separation, and paired comparison. A final scientific “GO” is not given unless the source/unit/quality review of the features is approved by the user.

In [ ]:
checks={
    'base_target_unique_city_year': not base.duplicated(['UrbanId','year']).any(),
    'external_feature_unique_city_year': not extf.duplicated(['UrbanId','year']).any(),
    'joined_rows_match_baseline': len(joined)==len(base),
    'primary_pool_nonempty': len(primary_pool)>0,
    'primary_external_columns_present': set(DEM_ERA5_FEATURES).issubset(joined.columns),
    'model_predictors_exclude_ID_country_target': not (set(BASELINE_FEATURES+DEM_ERA5_FEATURES+NDVI_FEATURES)&forbidden),
    'all_baseline_models_reported': {'source_mean','ridge','rf'}.issubset(set(results_all['model'])),
    'no_test_city_overlap': bool((results_all['test_city_overlap']==0).all()),
    'paired_comparison_created': len(comparison)>0 and not comparison.empty,
    'external_feature_coverage_report_created': (EXTERNAL_DIR/'external_feature_coverage_by_country.csv').exists(),
    'both_transfer_directions_reported': {'China_to_Turkiye','Turkiye_to_China'}.issubset(set(results_all['direction'])),
    'primary_pool_has_both_countries': {'China','Türkiye'}.issubset(set(primary_pool['Country'].unique())) and bool((primary_pool.groupby('Country')['UrbanId'].nunique()>=2).all()),
    'reported_error_metrics_finite': bool(np.isfinite(results_all[['rmse','mae']].to_numpy(dtype=float)).all()),
}
status='GO' if all(checks.values()) else 'NO-GO / REVIEW REQUIRED'
report=[
    '# Enhanced Transfer Decision Log','',f'- Run ID: `{RUN_ID}`',f'- Decision: **{status}**','',
    f'- Baseline target rows: {len(base):,}',f'- Primary common-pool rows: {len(primary_pool):,}',
    f'- NDVI complete-case rows: {len(ndvi_pool):,}',
    f'- Total model evaluations: {len(results_all):,}',
    f'- Paired comparison rows: {len(comparison):,}','','## Checks','',
    *([f"- [{'x' if ok else ' '}] {name}" for name,ok in checks.items()]),'',
    '## Interpretation','',
    '- Model results are matched within each pool, transfer direction, model, few-shot ratio, and seed.',
    '- Baseline-versus-extended comparisons are valid only within their named common pool and paired keys.',
    '- NDVI complete-case results represent a smaller, potentially selected sample; they should not be compared directly with primary-pool scores without acknowledging the population change.',
    '- The decision covers only the machine-checkable gates listed here. It is not causal evidence and does not mean raw LST pixels were recomputed.',
    '- NDVI is sampled at city coordinates; it is not a polygon-weighted urban-area mean.',
    '- Keep the external-feature QA report and country coverage tables with these model results.'
]
(DOCS_DIR/'DECISION_LOG_ENHANCED.md').write_text('\n'.join(report),encoding='utf-8')
(PHASE2_ROOT/'task7_partition_report.md').write_text('\n'.join(report),encoding='utf-8')
print('\n'.join(report))

# Enhanced Transfer Decision Log

- Run ID: `20261019_151310`
- Decision: **GO**

- Baseline target rows: 32,420
- Primary common-pool rows: 32,420
- NDVI complete-case rows: 18,237
- Total model evaluations: 1,530
- Paired comparison rows: 612

## Checks

- [x] base_target_unique_city_year
- [x] external_feature_unique_city_year
- [x] joined_rows_match_baseline
- [x] primary_pool_nonempty
- [x] primary_external_columns_present
- [x] model_predictors_exclude_ID_country_target
- [x] all_baseline_models_reported
- [x] no_test_city_overlap
- [x] paired_comparison_created
- [x] external_feature_coverage_report_created
- [x] both_transfer_directions_reported
- [x] primary_pool_has_both_countries
- [x] reported_error_metrics_finite

## Interpretation

- Model results are matched within each pool, transfer direction, model, few-shot ratio, and seed.
- Baseline-versus-extended comparisons are valid only within their named common pool and paired keys.
- NDVI complete-case results represent a sm

In [ ]:

# Check the project files before running the experiments.

from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

if not drive_root.exists():
    from google.colab import drive
    drive.mount("/content/drive")

project = (
    Path("/content/drive/MyDrive")
    / "Urban_Heat_Transfer_COMPLETE_BUNDLE"
    / "urban-heat-transfer"
)

base_file = (
    project / "outputs" / "phase2_outputs"
    / "tasks_feature_matrix.csv"
)

external_file = (
    project / "data" / "external_features"
    / "external_features.csv"
)

print("Project folder exists:", project.is_dir())
print("Notebook 1 input exists:", base_file.exists())
print("Notebook 2 output exists:", external_file.exists())
print("Project path:", project)


Mounted at /content/drive
Project folder exists: True
Notebook 1 input exists: False
Notebook 2 output exists: True
Project path: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer


In [ ]:

# Find the input file from Notebook 1.

from pathlib import Path

project = Path(
    "/content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer"
)

phase2 = project / "outputs" / "phase2_outputs"

print("Phase 2 folder exists:", phase2.is_dir())

if phase2.is_dir():
    print("\nFiles in phase2_outputs:")
    for path in sorted(phase2.iterdir()):
        print(path.name)

print("\nPossible feature matrix files:")
matches = []

for path in project.rglob("*.csv"):
    name = path.name.lower()
    if "feature" in name or "task" in name or "matrix" in name:
        matches.append(path)

for path in matches[:100]:
    print(path.relative_to(project))

if not matches:
    print("No matching CSV files found.")


Phase 2 folder exists: True

Files in phase2_outputs:
.gitkeep
README.md
analysis_city_manifest.csv
audit_reports
disputed_urbanid_review.csv
era5_jja_features.csv
era5_land_jja_2001_2020.nc
era5_land_jja_2001_2020_metadata.json
phase2_cleaned_target.csv
phase2_file_inventory.csv
phase2_primary_target_raw.csv
task3_composite_high_residual_review.csv
task3_composite_review_decisions.csv
task3_composite_validation.csv
task3_composite_validation_country_summary.csv
task3_composite_validation_report.md
task3_composite_validation_summary.csv
task3_duplicate_or_ambiguous_keys.csv
task4_coverage_summary.md
task4_missing_city_years.csv
task4_quarantined_summary.csv
task4_year_country_coverage.csv
task5_cleaning_log.csv
task5_cleaning_report.md
task5_country_year_summary.csv
task5_coverage_reconciliation.csv
task5_outlier_review.csv
task5_target_validation.csv
task6_feature_coverage_by_country.csv
task6_feature_dictionary.md
task6_feature_matrix.csv
task6_leakage_audit.csv
task6_vif_log.csv
tas

In [ ]:

# Check the project folder.

from pathlib import Path

root = Path("/content/drive/MyDrive")
project = root / "Urban_Heat_Transfer_COMPLETE_BUNDLE" / "urban-heat-transfer"

print("Project folder:", project.is_dir())
print("Notebooks folder:", (project / "notebooks").is_dir())
print("Data folder:", (project / "data").is_dir())
print("Baseline file:", (project / "outputs" / "phase2_outputs" / "task6_feature_matrix.csv").is_file())
print("External file:", (project / "data" / "external_features" / "external_features.csv").is_file())


Project folder: True
Notebooks folder: True
Data folder: True
Baseline file: True
External file: True


In [ ]:

# Check the enhanced feature matrix.

from pathlib import Path
import pandas as pd

project = Path(
    "/content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer"
)

feature_file = (
    project / "outputs" / "phase2_outputs"
    / "task6_feature_matrix_enhanced.csv"
)

print("File exists:", feature_file.is_file())

if feature_file.is_file():
    df = pd.read_csv(feature_file)

    print("Rows:", len(df))
    print("Cities:", df["UrbanId"].nunique())
    print("Years:", df["year"].nunique())
    print(
        "Duplicate city-year keys:",
        df.duplicated(["UrbanId", "year"]).sum()
    )

    print("\nMissing values in main features:")
    cols = [
        "Intensity_MEA",
        "elevation_m",
        "era5_t2m_jja",
        "era5_precip_jja_mm",
        "era5_wind_jja",
        "era5_dewpoint_jja",
        "era5_rh_jja",
        "ndvi_jja_mean",
    ]

    print(df[[c for c in cols if c in df.columns]].isna().sum())


File exists: True
Rows: 32420
Cities: 1621
Years: 20
Duplicate city-year keys: 0

Missing values in main features:
Intensity_MEA             0
elevation_m               0
era5_t2m_jja              0
era5_precip_jja_mm        0
era5_wind_jja             0
era5_dewpoint_jja         0
era5_rh_jja               0
ndvi_jja_mean         14183
dtype: int64


In [ ]:
# Load saved paired comparison results
comparison_path = PHASE2_ROOT / "baseline_vs_enhanced_comparison.csv"
paired = pd.read_csv(comparison_path)

print("Comparison rows:", len(paired))
print("Comparison types:", paired["comparison"].unique())

# Summarize the average change in model performance
summary = (
    paired[paired["model"].isin(["rf", "ridge"])]
    .groupby(
        ["comparison", "direction", "model", "few_shot_ratio"],
        as_index=False
    )
    .agg(
        mean_delta_r2=("delta_r2", "mean"),
        mean_delta_rmse=("delta_rmse", "mean"),
        mean_delta_mae=("delta_mae", "mean"),
        seed_count=("seed", "nunique")
    )
)

display(summary.round(3))

Comparison rows: 612
Comparison types: ['baseline_vs_dem_era5' 'dem_era5_vs_ndvi']


,comparison,direction,model,few_shot_ratio,mean_delta_r2,mean_delta_rmse,mean_delta_mae,seed_count
0,baseline_vs_dem_era5,China_to_Turkiye,rf,0.00,0.394,-0.134,-0.131,1
1,baseline_vs_dem_era5,China_to_Turkiye,rf,0.01,0.315,-0.112,-0.113,10
2,baseline_vs_dem_era5,China_to_Turkiye,rf,0.05,0.182,-0.069,-0.078,10
3,baseline_vs_dem_era5,China_to_Turkiye,rf,0.10,0.048,-0.018,-0.026,10
4,baseline_vs_dem_era5,China_to_Turkiye,rf,0.20,0.060,-0.023,-0.026,10
5,baseline_vs_dem_era5,China_to_Turkiye,rf,0.50,0.031,-0.012,-0.013,10
6,baseline_vs_dem_era5,China_to_Turkiye,ridge,0.00,0.517,-0.139,-0.150,1
7,baseline_vs_dem_era5,China_to_Turkiye,ridge,0.01,0.657,-0.192,-0.205,10
8,baseline_vs_dem_era5,China_to_Turkiye,ridge,0.05,0.457,-0.160,-0.163,10
9,baseline_vs_dem_era5,China_to_Turkiye,ridge,0.10,0.238,-0.092,-0.092,10


In [ ]:
# Check whether feature improvements are consistent across seeds

seed_check = (
    paired[paired["model"].isin(["rf", "ridge"])]
    .groupby(
        ["comparison", "direction", "model", "few_shot_ratio"],
        as_index=False
    )
    .agg(
        mean_delta_r2=("delta_r2", "mean"),
        std_delta_r2=("delta_r2", "std"),
        min_delta_r2=("delta_r2", "min"),
        max_delta_r2=("delta_r2", "max"),
        positive_seed_pct=(
            "delta_r2",
            lambda x: 100 * (x > 0).mean()
        ),
        seed_count=("seed", "nunique")
    )
)

display(seed_check.round(3))

,comparison,direction,model,few_shot_ratio,mean_delta_r2,std_delta_r2,min_delta_r2,max_delta_r2,positive_seed_pct,seed_count
0,baseline_vs_dem_era5,China_to_Turkiye,rf,0.00,0.394,NaN,0.394,0.394,100.0,1
1,baseline_vs_dem_era5,China_to_Turkiye,rf,0.01,0.315,0.148,-0.054,0.451,90.0,10
2,baseline_vs_dem_era5,China_to_Turkiye,rf,0.05,0.182,0.104,-0.031,0.275,90.0,10
3,baseline_vs_dem_era5,China_to_Turkiye,rf,0.10,0.048,0.143,-0.254,0.318,70.0,10
4,baseline_vs_dem_era5,China_to_Turkiye,rf,0.20,0.060,0.161,-0.173,0.350,60.0,10
5,baseline_vs_dem_era5,China_to_Turkiye,rf,0.50,0.031,0.112,-0.103,0.305,50.0,10
6,baseline_vs_dem_era5,China_to_Turkiye,ridge,0.00,0.517,NaN,0.517,0.517,100.0,1
7,baseline_vs_dem_era5,China_to_Turkiye,ridge,0.01,0.657,0.098,0.467,0.796,100.0,10
8,baseline_vs_dem_era5,China_to_Turkiye,ridge,0.05,0.457,0.097,0.337,0.571,100.0,10
9,baseline_vs_dem_era5,China_to_Turkiye,ridge,0.10,0.238,0.069,0.135,0.354,100.0,10


In [ ]:
# Review Random Forest performance across experiments

results_check = pd.read_csv(
    PHASE2_ROOT / "task7_transfer_results_all_variants.csv"
)

selected = results_check[
    (results_check["model"] == "rf") &
    (
        (
            (results_check["pool"] == "PRIMARY_COMMON_POOL") &
            (results_check["feature_set"].isin(
                ["geo_baseline", "dem_era5"]
            ))
        ) |
        (
            (results_check["pool"] == "NDVI_COMPLETE_SUBSET") &
            (results_check["feature_set"].isin(
                ["dem_era5", "dem_era5_ndvi"]
            ))
        )
    )
]

summary = (
    selected.groupby(
        ["pool", "direction", "feature_set", "few_shot_ratio"],
        as_index=False
    )
    .agg(
        mean_r2=("r2", "mean"),
        std_r2=("r2", "std"),
        min_r2=("r2", "min"),
        max_r2=("r2", "max"),
        mean_mae=("mae", "mean"),
        seed_count=("seed", "nunique")
    )
    .sort_values(
        ["pool", "direction", "few_shot_ratio", "feature_set"]
    )
)

display(summary.round(3))

,pool,direction,feature_set,few_shot_ratio,mean_r2,std_r2,min_r2,max_r2,mean_mae,seed_count
0,NDVI_COMPLETE_SUBSET,China_to_Turkiye,dem_era5,0.00,0.186,NaN,0.186,0.186,0.529,1
6,NDVI_COMPLETE_SUBSET,China_to_Turkiye,dem_era5_ndvi,0.00,0.280,NaN,0.280,0.280,0.506,1
1,NDVI_COMPLETE_SUBSET,China_to_Turkiye,dem_era5,0.01,0.251,0.060,0.185,0.381,0.509,10
7,NDVI_COMPLETE_SUBSET,China_to_Turkiye,dem_era5_ndvi,0.01,0.274,0.079,0.071,0.353,0.506,10
2,NDVI_COMPLETE_SUBSET,China_to_Turkiye,dem_era5,0.05,0.189,0.127,-0.058,0.321,0.516,10
8,NDVI_COMPLETE_SUBSET,China_to_Turkiye,dem_era5_ndvi,0.05,0.271,0.123,0.013,0.432,0.486,10
3,NDVI_COMPLETE_SUBSET,China_to_Turkiye,dem_era5,0.10,0.240,0.127,-0.070,0.368,0.501,10
9,NDVI_COMPLETE_SUBSET,China_to_Turkiye,dem_era5_ndvi,0.10,0.335,0.068,0.175,0.409,0.462,10
4,NDVI_COMPLETE_SUBSET,China_to_Turkiye,dem_era5,0.20,0.172,0.177,-0.216,0.398,0.515,10
10,NDVI_COMPLETE_SUBSET,China_to_Turkiye,dem_era5_ndvi,0.20,0.236,0.127,-0.061,0.387,0.486,10


## Results and Motivation for Further Feature Engineering

Transfer learning experiments evaluated whether urban heat intensity models learned in one country could be transferred to another, and whether limited target country training data and additional environmental features could improve performance.

The results showed **asymmetric transfer performance**. In the primary experiment using DEM and ERA5 features, the Random Forest model achieved an average R² value of 0.241 for transfer from Türkiye to China at a 50% adaptation level. The corresponding average R² value for transfer from China to Türkiye was 0.007.

NDVI-based experiments showed potential improvements in some settings. However, since these experiments used a smaller full case dataset, their results should not be directly compared to those obtained from the full primary dataset.

These findings encourage further investigation of additional urban features. In the next phase, population distribution, built-up surface, and land cover composition will be evaluated as potential predictors of urban heat intensity.

To maintain reproducibility, existing experiments and outputs will remain unchanged. New features and experiments will be developed in a separate notebook using comparable data pods and evaluation procedures wherever possible.

The aim of the next phase is to determine whether these additional features improve prediction performance in both transfer directions, rather than assuming that adding more data will necessarily improve the model.

In [3]:
print("BASELINE_FEATURES =", BASELINE_FEATURES)
print("DEM_ERA5_FEATURES =", DEM_ERA5_FEATURES)
print("NDVI_FEATURES =", NDVI_FEATURES)

print("TARGET_NAME =", TARGET_NAME)
print("FEW_SHOT_RATIOS =", FEW_SHOT_RATIOS)
print("ADAPTATION_SEEDS =", ADAPTATION_SEEDS)
print("TEST_LOCK_SEED =", TEST_LOCK_SEED)
print("TEST_FRACTION =", TEST_FRACTION)

BASELINE_FEATURES = ['year', 'Longitude', 'Latitude', 'area_log']
DEM_ERA5_FEATURES = ['elevation_m', 'era5_t2m_jja', 'era5_precip_jja_mm', 'era5_wind_jja', 'era5_dewpoint_jja', 'era5_rh_jja']
NDVI_FEATURES = ['ndvi_jja_mean']
TARGET_NAME = Intensity_MEA
FEW_SHOT_RATIOS = [0.0, 0.01, 0.05, 0.1, 0.2, 0.5]
ADAPTATION_SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
TEST_LOCK_SEED = 20261017
TEST_FRACTION = 0.2
